# Dunnhumby NGCF seed50·51 개인 Drive 재개
기존 결과를 덮어쓰지 않고 완료 모형은 재사용합니다. 검증된 최신 checkpoint에서 재개합니다. 기존 개인 NGCF와 같은 마운트 Drive 저장을 사용하며 학교 API 용량/업로드 wrapper를 적용하지 않습니다. 최신·직전 전체 상태를 보존합니다. 학습 source·run·seed·300epoch·평가 조건은 그대로입니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, subprocess, sys, importlib.util, urllib.request, torch
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0)
SOURCE='63146eff4ed60bfb0cc430b77d58fdc40e4db7c5'
REPO=Path('/content/clv-dh-ngcf-remaining9-'+SOURCE[:12])
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout',SOURCE],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE
sys.path.insert(0,str(REPO))
HELPER='46c9c6dbddad7ea384e267f2cbd17708707db994'
for name,sha in {'clv_dh_ngcf_split72':'96c0b3d9a08d2a244c1bc15ea1967fa3414b85f46754cdb69ad9bfeba2b5fb43','clv_ngcf_school_storage':'0d3be22fe7d604adabdf1aaffcfbfe7d1dc748cbba4b1c7073334e0c2bd1bd4c'}.items():
    with urllib.request.urlopen(f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{HELPER}/{name}.py',timeout=60) as response: data=response.read()
    assert hashlib.sha256(data).hexdigest()==sha
    target=Path('/content')/(name+'.py');target.write_bytes(data)
    spec=importlib.util.spec_from_file_location(name,target);module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
import clv_dh_ngcf_split72 as split
print('개인 Drive 마운트 저장 / L4 / 고정 학습 source 준비 완료')


In [ ]:
from zipfile import ZipFile
import shutil
ARCHIVE_SHA='f94f1af405567582e2f87fd8ab1632e754f3301181cf09fe9e9c8fbb08fc97aa'
archive=Path('/content/drive/MyDrive/dh_ngcf_seed50_51_migrated_20261027/dh_ngcf_school2_migration_20261027.zip')
assert archive.is_file(),'개인 Drive의 이전 폴더에서 이동 ZIP을 확인하세요.'
assert hashlib.sha256(archive.read_bytes()).hexdigest()==ARCHIVE_SHA
ROOT=Path('/content/ngcf_migration_import_20261027')
with ZipFile(archive) as z:
    names=z.namelist()
    assert len(names)==len(set(names)) and all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names)
    assert z.testzip() is None
    manifest=json.loads(z.read('manifest.json'))
    assert set(names)==set(manifest)|{'manifest.json','source_drive_files.json'}
    for name,sha in manifest.items(): assert hashlib.sha256(z.read(name)).hexdigest()==sha,name
    z.extractall(ROOT)
inputs=Path('/content/ngcf_school2_inputs')
with ZipFile(ROOT/'inputs.zip') as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(inputs)
for name,sha in json.loads((inputs/'manifest.json').read_text()).items():
    assert hashlib.sha256((inputs/name).read_bytes()).hexdigest()==sha
out=Path('/content/drive/MyDrive/논문/data/results_v3_dunnhumby_ngcf_school2_seed50_51_v1')
run=out/'ae1edba2afd9'
"""Repeatable migration: never replace existing destination files or newer epochs."""
from pathlib import Path
import hashlib
import os
import shutil


def import_missing(source, destination):
    source, destination = Path(source), Path(destination)
    # Check the immutable protocol before copying any missing file.
    protocol = destination / 'protocol.json'
    if protocol.exists() and protocol.read_bytes() != (source/'protocol.json').read_bytes():
        raise RuntimeError('Existing run protocol differs; nothing imported')
    copied, preserved = [], []
    for f in sorted(source.rglob('*')):
        if not f.is_file():
            continue
        rel = f.relative_to(source)
        dest = destination / rel
        if dest.exists():
            if not dest.is_file():
                raise RuntimeError(f'Expected file: {dest}')
            preserved.append(str(rel))
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        # Exclusive creation never overwrites a file created concurrently.
        try:
            with dest.open('xb') as out, f.open('rb') as inp:
                shutil.copyfileobj(inp, out)
                out.flush()
                os.fsync(out.fileno())
        except FileExistsError:
            preserved.append(str(rel))
            continue
        if hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(f.read_bytes()).digest():
            raise RuntimeError(f'Imported file hash mismatch: {dest}')
        copied.append(str(rel))
    return {'copied': copied, 'preserved': preserved}


def pending_epochs(remote_rows, archive_rows, *, completed):
    """Never republish an old packet over a resumed run; retain remote top two."""
    if completed:
        return []
    current = max((int(r['appProperties']['clv_epoch']) for r in remote_rows), default=0)
    for row in archive_rows:
        p = row['appProperties']
        for live in remote_rows:
            q = live['appProperties']
            if int(p['clv_epoch']) == int(q['clv_epoch']) and p['clv_sha256'] != q['clv_sha256']:
                raise RuntimeError('Same epoch has different checkpoint bytes; originals preserved')
    return sorted([r for r in archive_rows if int(r['appProperties']['clv_epoch']) > current],
                  key=lambda r: int(r['appProperties']['clv_epoch']))

import_report=import_missing(ROOT/'run',run)
print('Missing files imported:',len(import_report['copied']),'Existing files preserved:',len(import_report['preserved']))


In [ ]:
import clv_dh_ngcf_remaining9 as screen
from dataclasses import asdict
assert Path(screen.__file__).resolve().parent==REPO.resolve()
assert Path(screen.original.__file__).resolve().parent==REPO.resolve()
screen.base.v3.SCHEMA['dunnhumby']['tx_path']=str(inputs/'raw/dunnhumby/transaction_data.csv')
screen.base.v3.SCHEMA['dunnhumby']['item_meta_path']=str(inputs/'raw/dunnhumby/product.csv')
cfg=screen.configure(out_dir=str(out),m2_reference=str(inputs/'references/clv_m2_adaptive_dunnhumby_final10_results.zip'),m4_reference=str(inputs/'references/clv_m4_user_centered_final10_results.zip'),seed49_reference=str(inputs/'references/dh_ngcf_final8_seed49_results.zip'))
prepared=screen.prepare(cfg)
assert prepared['config_hash']=='ae1edba2afd9'
assert prepared['protocol']==json.loads((ROOT/'run/protocol.json').read_text())
split.check_personal_receipt(prepared,ROOT/'personal7_assignment.json')
split.install(prepared,'school')
for seed in (50,51):
    for label in screen.ORDER:
        screen.completed(prepared,label,seed)  # validates existing completed identities/results
print('Current completion:',split.status(prepared,'school'))
"""Use the established personal mounted-Drive save path; no quota/API wrapper."""
from dataclasses import asdict
from pathlib import Path
import hashlib
import os
import shutil
import torch


def checked(path, ident):
    p=torch.load(path,map_location='cpu',weights_only=False)
    if p.get('identity')!=asdict(ident): raise RuntimeError('Checkpoint identity mismatch')
    ep=p.get('epoch')
    if not isinstance(ep,int) or not 1<=ep<=300 or [r['epoch'] for r in p['history']]!=list(range(1,ep+1)):
        raise RuntimeError('Checkpoint history mismatch')
    for k in ('model_state','optimizer_state','numpy_rng_state','torch_rng_state','cuda_rng_state'):
        if k not in p: raise RuntimeError('Missing training state: '+k)
    return ep


def copy_verified(source,target):
    target.parent.mkdir(parents=True,exist_ok=True)
    temp=target.with_suffix(target.suffix+'.copytmp')
    with source.open('rb') as src,temp.open('wb') as dst:
        shutil.copyfileobj(src,dst);dst.flush();os.fsync(dst.fileno())
    if hashlib.sha256(source.read_bytes()).digest()!=hashlib.sha256(temp.read_bytes()).digest():
        raise RuntimeError('Checkpoint copy hash mismatch')
    os.replace(temp,target)


def install(screen, prepared, mydrive='/content/drive/MyDrive', local_root='/content/ngcf_school_local'):
    base=getattr(screen.original,'_school_original_store',screen.original.SafeProgressStore)
    if getattr(base,'_personal_mounted',False): return
    run=Path(prepared['run_dir'])
    remote=Path(mydrive)/('ngcf_school2_checkpoints_'+prepared['config_hash'])
    if not Path(mydrive).is_dir(): raise RuntimeError('Personal Drive mount required')
    class PersonalStore(base):
        _personal_mounted=True
        def __init__(self,root,ident):
            if Path(root)!=run/'progress' or ident.seed not in (50,51): raise RuntimeError('Wrong run/seed')
            super().__init__(root,ident)
        def restore_epoch(self,model,optimizer,rng):
            local=Path(local_root)/self.identity.config_hash/self.latest_checkpoint.name
            verified_local=Path('/content/ngcf_verified_recovery')/self.identity.config_hash/self.latest_checkpoint.name
            candidates=[p for p in (self.latest_checkpoint,self.latest_checkpoint.with_suffix('.previous.pt'),local,verified_local) if p.exists()]
            candidates+=list(remote.glob(f'{self.identity.model_id}_s{self.identity.seed}_ep*.pt'))
            ranked=[]; invalid=[]
            for p in dict.fromkeys(candidates):
                try: ep=checked(p,self.identity)
                except (EOFError,OSError) as exc:
                    invalid.append(p); print('[읽기 실패·원본 보존]',p.name,type(exc).__name__,flush=True); continue
                ranked.append((ep,str(p),p))
            ranked.sort(reverse=True)
            if invalid and not ranked: raise RuntimeError('정상 복구본 없음: 자동 처음부터 학습 금지')
            from datetime import datetime, timezone
            for p in invalid:
                if p in (self.latest_checkpoint,self.latest_checkpoint.with_suffix('.previous.pt')):
                    p.rename(p.with_name(p.name+'.invalid-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')))
            if ranked:
                ep,_,latest=ranked[0]
                previous=next((p for e,_,p in ranked if e<ep),None)
                prevpath=self.latest_checkpoint.with_suffix('.previous.pt')
                if previous is not None and not prevpath.exists(): copy_verified(previous,prevpath)
                if latest!=self.latest_checkpoint: copy_verified(latest,self.latest_checkpoint)
                print(f'[개인 Drive 복원] epoch {ep}; API 저장·용량조회 제약 없음',flush=True)
            return super().restore_epoch(model,optimizer,rng)
        def save_epoch(self,model,optimizer,rng,**state):
            if self.latest_checkpoint.exists():
                old=checked(self.latest_checkpoint,self.identity)
                if state['epoch']<=old: raise RuntimeError('Refuse checkpoint rollback')
                copy_verified(self.latest_checkpoint,self.latest_checkpoint.with_suffix('.previous.pt'))
            durable=self.latest_checkpoint
            local=Path('/content/ngcf_verified_recovery')/self.identity.config_hash/durable.name
            local.parent.mkdir(parents=True,exist_ok=True)
            try:
                self.latest_checkpoint=local
                super().save_epoch(model,optimizer,rng,**state)
            finally: self.latest_checkpoint=durable
            assert checked(local,self.identity)==state['epoch']
            copy_verified(local,durable)
            assert checked(durable,self.identity)==state['epoch']
            self._write_progress('running',{'epoch':int(state['epoch']),'max_epoch':300,'selection':'none','checkpoint_sha256':hashlib.sha256(local.read_bytes()).hexdigest(),'checkpoint_bytes':local.stat().st_size})
            result=durable
            print(f'[개인 Drive 저장] epoch {state["epoch"]}: 최신·직전 유지',flush=True)
            return result
    screen.original.SafeProgressStore=PersonalStore
    print('기존 개인 NGCF 방식: 마운트 Drive 매epoch 저장. 학교 API/고정여유/재시도 wrapper 해제.',flush=True)

install(screen,prepared)


In [ ]:
# seed50/51 only; completed models are reused and latest durable state restored.
try:
    result=split.run(prepared,'school')
except Exception:
    import traceback
    traceback.print_exc()
    print('저장 확인 실패로 중단. 기존 결과와 최신/직전 checkpoint 보존.', flush=True)
else:
    packet=split.export_partition(prepared,'school','/content/drive/MyDrive/dh_ngcf_school2_results.zip')
    print('50/51 완료 결과 패킷:',packet)
    print(json.dumps(result,ensure_ascii=False,indent=2))
